In [92]:
# Import libraries
import numpy as np
import pandas as pd
import warnings 
warnings.filterwarnings('ignore')

In [93]:
# Import datasets 
df = pd.read_csv("Cleaned_data.csv")

In [94]:
df.columns

Index(['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level', 'diabetes'],
      dtype='str')

In [95]:
# Divide data into X and y 
X = df[['gender', 'age', 'hypertension', 'heart_disease', 'smoking_history',
       'bmi', 'HbA1c_level', 'blood_glucose_level']]
y = df['diabetes']

In [96]:
# Label encoding
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
X['gender'] = le.fit_transform(X['gender'])
X['smoking_history'] = le.fit_transform(X['smoking_history'])

In [97]:
# Divide into train and test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.1, random_state=0, stratify = y)

In [98]:
from imblearn.over_sampling import SMOTE
smote = SMOTE(random_state = 42)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

In [89]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    random_state=42
)
model.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['diabetes'], result['Prediction'])
score = accuracy_score(result['diabetes'], result['Prediction'])
print(cm)
print(score)

[[7855  864]
 [ 108  740]]
0.8984007525870179


In [99]:
from xgboost import XGBClassifier

model1 = XGBClassifier(
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model1.fit(X_resampled, y_resampled)

#evaluate model
from sklearn.metrics import confusion_matrix, accuracy_score
y_pred = model1.predict(X_test)
y_pred = pd.DataFrame(data = y_pred, columns = ['Prediction'])
result = pd.concat([y_test.reset_index(drop = True), y_pred], axis = 1)
cm = confusion_matrix(result['diabetes'], result['Prediction'])
score = accuracy_score(result['diabetes'], result['Prediction'])
print(cm)
print(score)

[[8651   68]
 [ 213  635]]
0.9706282011079753


In [101]:
#accuracy score of random forest algo is large so choose to export this model
import pickle
pickle.dump(model1, open('model.pkl', 'wb'))
pickle.dump(sc, open('sc.pkl', 'wb'))